In [ ]:
# ANA-GIA (Analytics-based Gradient Inversion Attack)
import torch.nn as nn

torch.manual_seed(0)

# THE VICTIM'S PRIVATE BATCH
input_dim  = 64      # e.g. flattened 8x8 "image"
batch_size = 6        # number of private samples in this training step
private_batch = torch.rand(batch_size, input_dim)

print("=" * 70)
print(f"STEP 0: {batch_size} private inputs (hidden from attacker)")
print("=" * 70)
for i in range(batch_size):
    print(f"--- sample {i} ---")
    print(private_batch[i].view(8, 8))
# STEP 1 — MALICIOUS MODEL SETUP (attacker crafts this BEFORE any
# gradients are received)
# num_bins controls resolution: more bins = narrower intervals = higher
# chance each interval contains exactly one sample. Rule of thumb:
# use several times more bins than batch_size.
num_bins = 400

class ImprintModel(nn.Module):
    """
    fc1: the malicious 'imprint' layer -- every neuron shares the same
         weight direction, with thresholds (biases) staggered to form
         `num_bins` increasing cutoffs.
    agg: a FIXED, equal-weight summation over all neurons. This keeps
         the per-sample gradient scaling factor (dL/dh_k) identical
         across every neuron k, which is what makes bin-subtraction
         give an exact result instead of a distorted mixture.
    """
    def __init__(self, input_dim, num_bins):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, num_bins)
        self.relu = nn.ReLU()
        self.agg = nn.Linear(num_bins, 1, bias=False)

    def forward(self, x):
        h = self.relu(self.fc1(x))
        return self.agg(h)

model = ImprintModel(input_dim, num_bins)

with torch.no_grad():
    direction = torch.randn(input_dim)
    direction = direction / direction.norm()
    model.fc1.weight.copy_(direction.repeat(num_bins, 1))

    # Estimate the plausible range of projections from a public prior
    # (attacker doesn't need the real batch, just a rough sense of the
    # input distribution -- e.g. from public data of the same domain).
    prior = torch.rand(5000, input_dim)
    proj_prior = prior @ direction
    thresholds = torch.quantile(proj_prior, torch.linspace(0.01, 0.99, num_bins))
    model.fc1.bias.copy_(-thresholds)   # neuron k fires when proj > thresholds[k]

    # Fixed equal-weight aggregation (this is what the victim's model
    # is tricked into believing is a normal layer).
    model.agg.weight.fill_(1.0)

print()
print("=" * 70)
print(f"Malicious bias thresholds (first 10 of {num_bins} bins shown)")
print("=" * 70)
print(thresholds[:10])
print("...")
print(thresholds[-10:])

# STEP 2 — VICTIM computes forward/backward pass on the PRIVATE batch
# and sends only the resulting gradients (this is the only leak).
model.zero_grad()
output = model(private_batch)          # shape [batch_size, 1]
loss = output.sum()                    # stand-in for a real training loss;
                                        # only its gradient matters here
loss.backward()

leaked_grad_W = model.fc1.weight.grad.clone()   # [num_bins, input_dim]
leaked_grad_b = model.fc1.bias.grad.clone()     # [num_bins]

# STEP 3 — ATTACK: bin-differencing reconstruction
# (uses ONLY leaked_grad_W / leaked_grad_b from here on)
eps = 1e-9
recovered_samples = []
recovered_bin_info = []   # (bin_index, threshold_low, threshold_high, diff_b value)

for k in range(num_bins - 1):
    diff_W = leaked_grad_W[k] - leaked_grad_W[k + 1]
    diff_b = leaked_grad_b[k] - leaked_grad_b[k + 1]
    if diff_b.abs() > eps:
        # This is the entry-wise division x = diff_W (elementwise) / diff_b (scalar)
        recovered_x = diff_W / diff_b
        recovered_samples.append(recovered_x)
        recovered_bin_info.append((k, thresholds[k].item(), thresholds[k + 1].item(), diff_b.item()))

recovered_samples = torch.stack(recovered_samples)  # [num_recovered, input_dim]

print()
print("=" * 70)
print(f"STEP 3: Attacker isolated {recovered_samples.shape[0]} candidate "
      f"reconstructions from {num_bins} bins")
print("=" * 70)

for idx, (bin_idx, t_low, t_high, db) in enumerate(recovered_bin_info):
    print(f"\n--- candidate {idx}  (bin #{bin_idx}, threshold range "
          f"[{t_low:.4f}, {t_high:.4f}], diff_b = {db:.6f}) ---")
    print("recovered_x = diff_W / diff_b  ->")
    print(recovered_samples[idx].view(8, 8))
# STEP 4 — Match each recovered candidate to its closest true sample
# and report accuracy (this comparison is only for OUR verification;
# a real attacker doesn't need `private_batch` to use the recovered data)
print()
print("=" * 70)
print("STEP 4: Matching recovered candidates to true private samples")
print("=" * 70)

matched = {}
for cand in recovered_samples:
    dists = (private_batch - cand).norm(dim=1)
    best_idx = dists.argmin().item()
    best_err = dists.min().item()
    # keep only the best (lowest-error) candidate per true sample
    if best_idx not in matched or best_err < matched[best_idx][1]:
        matched[best_idx] = (cand, best_err)

for i in range(batch_size):
    if i in matched:
        cand, err = matched[i]
        mse = torch.mean((cand - private_batch[i]) ** 2).item()
        print(f"sample {i}: RECOVERED  |  L2 error = {err:.6f}  |  MSE = {mse:.10f}")
    else:
        print(f"sample {i}: NOT recovered (no bin isolated it uniquely)")

recovered_count = len(matched)
print()
print(f"Recovered {recovered_count} / {batch_size} private samples exactly.")
# STEP 5 — Final side-by-side: true input vs. recovered input
print()
print("=" * 70)
print("STEP 5: Final comparison -- TRUE vs RECOVERED")
print("=" * 70)
for i in range(batch_size):
    print(f"\n########## sample {i} ##########")
    if i in matched:
        cand, err = matched[i]
        print("TRUE input:")
        print(private_batch[i].view(8, 8))
        print("RECOVERED input (from gradients only):")
        print(cand.view(8, 8))
    else:
        print("Not recovered.")
if recovered_count < batch_size:
    print("Tip: increase `num_bins` for finer resolution, or use a batch")
    print("     size smaller relative to num_bins to raise the recovery rate.")

STEP 0: 6 private inputs (hidden from attacker)
--- sample 0 ---
tensor([[0.4963, 0.7682, 0.0885, 0.1320, 0.3074, 0.6341, 0.4901, 0.8964],
        [0.4556, 0.6323, 0.3489, 0.4017, 0.0223, 0.1689, 0.2939, 0.5185],
        [0.6977, 0.8000, 0.1610, 0.2823, 0.6816, 0.9152, 0.3971, 0.8742],
        [0.4194, 0.5529, 0.9527, 0.0362, 0.1852, 0.3734, 0.3051, 0.9320],
        [0.1759, 0.2698, 0.1507, 0.0317, 0.2081, 0.9298, 0.7231, 0.7423],
        [0.5263, 0.2437, 0.5846, 0.0332, 0.1387, 0.2422, 0.8155, 0.7932],
        [0.2783, 0.4820, 0.8198, 0.9971, 0.6984, 0.5675, 0.8352, 0.2056],
        [0.5932, 0.1123, 0.1535, 0.2417, 0.7262, 0.7011, 0.2038, 0.6511]])
--- sample 1 ---
tensor([[0.7745, 0.4369, 0.5191, 0.6159, 0.8102, 0.9801, 0.1147, 0.3168],
        [0.6965, 0.9143, 0.9351, 0.9412, 0.5995, 0.0652, 0.5460, 0.1872],
        [0.0340, 0.9442, 0.8802, 0.0012, 0.5936, 0.4158, 0.4177, 0.2711],
        [0.6923, 0.2038, 0.6833, 0.7529, 0.8579, 0.6870, 0.0051, 0.1757],
        [0.7497, 0.6047, 0.11